In [1]:
from bs4 import BeautifulSoup
import html2text

In [2]:
import re
import os
import requests
import html2text
from bs4 import BeautifulSoup
from collections import deque

pattern = r"https?:\/\/(?:www\d*\.)?eecs\.berkeley\.edu(?:\/[^\s]*)?"

os.makedirs("crawled", exist_ok=True)

h2t = html2text.HTML2Text()
h2t.ignore_links = True
h2t.ignore_images = True
h2t.ignore_tables = False
h2t.body_width = 0


def clean_html_for_rag(html):
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup([
        "script", "style", "noscript",
        "nav", "footer", "header",
        "form", "input", "button", "select", "textarea",
        "svg", "img"
    ]):
        tag.decompose()

    noise_classes = [
        "sidebar", "ad", "ads", "promo",
        "cookie", "banner", "popup",
        "modal", "social", "share",
        "related", "footer"
    ]

    for tag in soup.find_all(True):
        classes = " ".join(tag.attrs.get("class", [])) if tag.attrs else ""
        if any(n in classes.lower() for n in noise_classes):
            tag.decompose()

    return h2t.handle(str(soup))


def get_matching_urls(html, base_url=None):
    soup = BeautifulSoup(html, "html.parser")
    urls = []

    for a in soup.find_all("a", href=True):
        href = a["href"]

        if href.startswith("/") and base_url:
            href = base_url.rstrip("/") + href

        if re.match(pattern, href):
            urls.append(href)

    return urls


def crawl_website(html, url):
    if "Forbidden" in html:
        return

    text = clean_html_for_rag(html)

    safe_name = re.sub(r"[^\w]", "_", url)

    with open(f"crawled/{safe_name}.txt", "w") as f:
        f.write(text)


seed = "https://eecs.berkeley.edu"
visited = set()
queue = deque([seed])
max_pages = 10


while queue and len(visited) < max_pages:
    url = queue.popleft()

    if url in visited:
        continue

    visited.add(url)
    print(f"Visiting ({len(visited)}/{max_pages}): {url}")

    try:
        response = requests.get(url, timeout=5)
    except:
        continue

    crawl_website(response.text, url)

    new_urls = get_matching_urls(response.text, base_url=url)

    for u in new_urls:
        if u not in visited:
            queue.append(u)


print(f"\nDone. Crawled {len(visited)} pages.")

Visiting (1/10): https://eecs.berkeley.edu
Visiting (2/10): https://eecs.berkeley.edu/resources/students/
Visiting (3/10): https://eecs.berkeley.edu/resources/faculty-staff/
Visiting (4/10): https://eecs.berkeley.edu/industry/
Visiting (5/10): https://eecs.berkeley.edu/latest-news/
Visiting (6/10): https://eecs.berkeley.edu/events/
Visiting (7/10): https://eecs.berkeley.edu/connect/support/
Visiting (8/10): https://eecs.berkeley.edu/about/
Visiting (9/10): https://eecs.berkeley.edu/about/history/
Visiting (10/10): https://eecs.berkeley.edu/about/history/first-women/

Done. Crawled 10 pages.
